<a href="https://colab.research.google.com/github/Matrioshka/TARA/blob/main/Habermas_Machine_DeepMind_example_aistudio_adapted_AF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemini-based Habermas Machine via AI Studio

The purpose of this notebook is to demonstrate the workings of the Habermas Machine, using publicly accessible models rather than the custom fine-tuned model. The Habermas Machine was introduced in this paper:

Tessler, M. H., Bakker, M. A., Jarrett D., Sheahan, H., Chadwick, M. J., Koster, R., Evans, G., Campbell-Gillingham, J., Collins, T., Parkes, D. C., Botvinick, M., and Summerfield, C. "AI can help humans find common ground in democratic deliberation." Science. (2024).

All parameters can be adjusted but as default this example uses:

*   Gemini 1.5 Flash via AIStudio OR OpenAI (AF)
*   Chain-of-thought statement and ranking model
*   5 participants and 4 candidate statements
*   Example data from Table S50 (p. 265) in the Supplementary Materials.
*   The Habermas Machine is set to verbose mode.

In this example, we overwrite the winner from the opinion round with the winner in the actual experiment to demonstrate how the critiquing works from our real example. However, when you use your own opinions and critiques, this can be turned off.

# Setup and Importing Packages

In [70]:
# Install dependencies from github.
# !pip install --upgrade git+https://github.com/google-deepmind/habermas_machine.git

import os
import re
import numpy as np
import google.generativeai as genai
from google.colab import userdata
try:
  from openai import OpenAI
except:
  !pip install -U openai
  from openai import OpenAI

try:
  from habermas_machine import machine
except: # If error, check that you have pip installed `habermas_machine`.
  !pip install --upgrade git+https://github.com/google-deepmind/habermas_machine.git
  from habermas_machine import machine

from habermas_machine import types
from habermas_machine.llm_client.aistudio_client import AIStudioClient
from habermas_machine.reward_model import base_model as reward_base_model
from habermas_machine.social_choice import utils as sc_utils


# API key
# api_key = userdata.get("GEMINI_API_KEY")
# if not api_key:
#     raise ValueError("Could not read GEMINI_API_KEY from Colab Secrets.")

api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Could not read OPENAI_API_KEY from Colab Secrets.")

client = OpenAI(api_key=api_key)
response = client.responses.create(
    model="gpt-5.4-mini",
    input="Reply with only: OK",
    max_output_tokens=20,
)

print(response.output_text)

os.environ["OPENAI_API_KEY"] = api_key
genai.configure(api_key=api_key)



OK


In [74]:
class OpenAIResponsesClient(base_client.LLMClient):
    """OpenAI Responses API adapter for Habermas Machine."""

    def __init__(self, model_name: str, api_key: str):
        if "gemini" in model_name.lower():
            raise ValueError(
                f"You passed a Gemini model name to the OpenAI client: {model_name}"
            )

        self.model_name = model_name
        self.client = OpenAI(api_key=api_key)

    def sample_text(
        self,
        prompt: str,
        *,
        max_tokens: int = 512,
        terminators=(),
        temperature: float = 0.2,
        timeout: float = 60,
        seed=None,
    ) -> str:
        del seed, terminators

        try:
            response = self.client.with_options(timeout=timeout).responses.create(
                model=self.model_name,
                input=prompt,
                max_output_tokens=max_tokens,
                temperature=temperature,
            )
        except Exception:
            # Some models/settings may reject temperature. Retry with minimal params.
            try:
                response = self.client.with_options(timeout=timeout).responses.create(
                    model=self.model_name,
                    input=prompt,
                    max_output_tokens=max_tokens,
                )
            except Exception as e2:
                print(f"OpenAI call failed: {type(e2).__name__}: {e2}")
                return ""

        text = getattr(response, "output_text", "") or ""

        if not text:
            parts = []
            for item in getattr(response, "output", []) or []:
                for content in getattr(item, "content", []) or []:
                    value = getattr(content, "text", "")
                    if value:
                        parts.append(value)
            text = "\n".join(parts)

        return text.strip()




In [72]:
class StableAIStudioClient(AIStudioClient):
    """A less brittle AI Studio client for this demo notebook."""

    def sample_text(
        self,
        prompt: str,
        *,
        max_tokens: int = 4096,
        terminators=(),
        temperature: float = 0.8,
        timeout: float = 60,
        seed=None,
    ) -> str:
        # Ignore terminators so Gemini can actually emit </answer>.
        # Lower temperature improves template-following.
        return super().sample_text(
            prompt,
            max_tokens=max(max_tokens, 8192),
            terminators=(),
            temperature=0.1,
            timeout=timeout,
            seed=seed,
        )


class FinalOnlyRankingModel(reward_base_model.BaseRankingModel):
    """Simpler ranking model: asks only for the final ranking, then parses it."""

    def predict_ranking(
        self,
        llm_client,
        question,
        opinion,
        statements,
        previous_winner=None,
        critique=None,
        seed=None,
        num_retries_on_error=None,
    ):
        if num_retries_on_error is None:
            num_retries_on_error = 8

        n = len(statements)
        letters = [chr(ord("A") + i) for i in range(n)]

        statements_text = "\n".join(
            f"{letter}. {statement.strip()}"
            for letter, statement in zip(letters, statements)
        )

        extra = ""
        if previous_winner is not None:
            extra += f"\nPrevious winning statement:\n{previous_winner}\n"
        if critique is not None:
            extra += f"\nParticipant critique:\n{critique}\n"

        prompt = f"""
You are ranking candidate consensus statements according to how well they match one participant's opinion.

Question:
{question}

Participant opinion:
{opinion}
{extra}

Candidate statements:
{statements_text}

Return exactly this format:

<answer>
One brief sentence explaining the ranking.
<sep>
A > B > C
</answer>

Rules:
- Use only the candidate letters: {", ".join(letters)}
- Include every candidate letter exactly once.
- Use ">" only.
- No ties.
- After <sep>, write only the final ranking.
- The ranking must contain exactly {n} letters.
""".strip()

        pattern = r"\b([A-Z](?:\s*>\s*[A-Z]){" + str(n - 1) + r"})\b"

        last_response = ""

        for attempt in range(num_retries_on_error + 1):
            response = llm_client.sample_text(
                prompt,
                max_tokens=1024,
                terminators=(),
                temperature=0.0,
                seed=seed,
            )
            last_response = response

            # Prefer the text after <sep>, but fall back to the whole response.
            search_area = response.split("<sep>", 1)[-1] if "<sep>" in response else response
            match = re.search(pattern, search_area)

            if not match:
                continue

            ranked_letters = re.findall(r"[A-Z]", match.group(1))

            if sorted(ranked_letters) != letters:
                continue

            ranking = np.empty(n, dtype=int)
            for rank, letter in enumerate(ranked_letters):
                ranking[ord(letter) - ord("A")] = rank

            return reward_base_model.RankingResult(ranking, response)

        return reward_base_model.RankingResult(
            None,
            f"FAILED_TO_PARSE_RANKING: {last_response}",
        )


class RobustHabermasMachine(machine.HabermasMachine):
    """Discard empty or malformed generated candidate statements."""

    def _generate_statements(self):
        statements = []
        explanations = []
        seen = set()

        max_attempts = max(20, self._num_candidates * max(5, self._num_retries_on_error or 1))

        for attempt in range(max_attempts):
            if len(statements) >= self._num_candidates:
                break

            indices = self._rng.permutation(self._num_citizens)
            shuffled_opinions = [self._opinions[j] for j in indices]

            shuffled_critiques = (
                [self._critiques[-1][j] for j in indices]
                if self._critiques
                else None
            )

            statement, explanation = self._statement_model.generate_statement(
                llm_client=self._statement_client,
                question=self._question,
                opinions=shuffled_opinions,
                previous_winner=(
                    self._previous_winners[-1]
                    if self._previous_winners
                    else None
                ),
                critiques=shuffled_critiques,
                seed=self._get_new_seed(),
                num_retries_on_error=self._num_retries_on_error,
            )

            statement = (statement or "").strip()

            if len(statement) > 20 and statement not in seen:
                statements.append(statement)
                explanations.append(explanation)
                seen.add(statement)
            else:
                print(f"Discarded bad candidate on attempt {attempt + 1}: {explanation}")

        if len(statements) < 2:
            raise ValueError(
                "Could not generate at least two valid candidate statements. "
                "Try a stronger model, fewer/lower-detail opinions, or more retries."
            )

        # If we could not get the requested number, downgrade cleanly.
        self._num_candidates = len(statements)

        return statements, explanations

In [73]:
# @title
# from habermas_machine.llm_client import base_client
# from habermas_machine.statement_model import base_model as statement_base_model

# class FastAIStudioClient(base_client.LLMClient):
#     """Small, timeout-aware Gemini client for the Habermas demo."""

#     def __init__(self, model_name: str):
#         self.model_name = model_name
#         self.model = genai.GenerativeModel(model_name=model_name)

#     def sample_text(
#         self,
#         prompt: str,
#         *,
#         max_tokens: int = 768,
#         terminators=(),
#         temperature: float = 0.2,
#         timeout: float = 45,
#         seed=None,
#     ) -> str:
#         del seed

#         try:
#             response = self.model.generate_content(
#                 prompt,
#                 generation_config=genai.GenerationConfig(
#                     temperature=temperature,
#                     max_output_tokens=max_tokens,
#                     stop_sequences=[],
#                 ),
#                 stream=False,
#                 request_options={"timeout": timeout},
#             )

#             try:
#                 return response.text.strip()
#             except Exception:
#                 parts = []
#                 for candidate in getattr(response, "candidates", []) or []:
#                     content = getattr(candidate, "content", None)
#                     for part in getattr(content, "parts", []) or []:
#                         text = getattr(part, "text", "")
#                         if text:
#                             parts.append(text)
#                 return "\n".join(parts).strip()

#         except Exception as e:
#             print(f"Gemini call failed or timed out: {type(e).__name__}: {e}")
#             return ""


# class FinalOnlyStatementModel(statement_base_model.BaseStatementModel):
#     """Generate only the final consensus statement, no chain-of-thought template."""

#     def generate_statement(
#         self,
#         llm_client,
#         question,
#         opinions,
#         previous_winner=None,
#         critiques=None,
#         seed=None,
#         num_retries_on_error=2,
#     ):
#         opinions_text = "\n".join(
#             f"Opinion {i + 1}: {opinion}" for i, opinion in enumerate(opinions)
#         )

#         extra = ""
#         if previous_winner is not None:
#             extra += f"\nPrevious winning statement:\n{previous_winner}\n"

#         if critiques is not None:
#             critiques_text = "\n".join(
#                 f"Critique {i + 1}: {critique}" for i, critique in enumerate(critiques)
#             )
#             extra += f"\nCritiques:\n{critiques_text}\n"

#         prompt = f"""
# You are helping a citizens' jury form a consensus statement.

# Question:
# {question}

# Individual opinions:
# {opinions_text}
# {extra}

# Write one concise consensus statement.

# Rules:
# - Do not provide step-by-step reasoning.
# - Do not use XML tags.
# - Do not mention that you are an AI.
# - Do not claim unanimity unless all opinions support it.
# - Preserve major disagreements and qualifications.
# - Maximum 180 words.

# Return only the consensus statement.
# """.strip()

#         last_response = ""

#         for attempt in range((num_retries_on_error or 0) + 1):
#             response = llm_client.sample_text(
#                 prompt,
#                 max_tokens=512,
#                 temperature=0.2,
#                 timeout=45,
#                 seed=seed,
#             ).strip()

#             last_response = response

#             # Clean common wrapper cruft if the model adds it anyway.
#             statement = re.sub(r"^```.*?\n|\n```$", "", response, flags=re.DOTALL).strip()
#             statement = statement.replace("<answer>", "").replace("</answer>", "").strip()
#             if "<sep>" in statement:
#                 statement = statement.split("<sep>")[-1].strip()

#             if len(statement) > 40:
#                 return statement_base_model.StatementResult(statement, "")

#         return statement_base_model.StatementResult(
#             last_response,
#             "FAILED_TO_GENERATE_STATEMENT",
#         )


# class FinalOnlyRankingModel(reward_base_model.BaseRankingModel):
#     """Ask only for final ranking, then parse it."""

#     def predict_ranking(
#         self,
#         llm_client,
#         question,
#         opinion,
#         statements,
#         previous_winner=None,
#         critique=None,
#         seed=None,
#         num_retries_on_error=2,
#     ):
#         n = len(statements)
#         letters = [chr(ord("A") + i) for i in range(n)]

#         statements_text = "\n".join(
#             f"{letter}. {statement.strip()}"
#             for letter, statement in zip(letters, statements)
#         )

#         prompt = f"""
# Rank the candidate consensus statements according to how well they match this participant's opinion.

# Question:
# {question}

# Participant opinion:
# {opinion}

# Candidate statements:
# {statements_text}

# Return only the ranking.

# Rules:
# - Use only these letters: {", ".join(letters)}
# - Include every letter exactly once.
# - Use ">" only.
# - No explanation.
# - No ties.

# Example:
# A > C > B
# """.strip()

#         pattern = r"\b([A-Z](?:\s*>\s*[A-Z]){" + str(n - 1) + r"})\b"
#         last_response = ""

#         for attempt in range((num_retries_on_error or 0) + 1):
#             response = llm_client.sample_text(
#                 prompt,
#                 max_tokens=64,
#                 temperature=0.0,
#                 timeout=30,
#                 seed=seed,
#             )

#             last_response = response
#             match = re.search(pattern, response)

#             if not match:
#                 continue

#             ranked_letters = re.findall(r"[A-Z]", match.group(1))

#             if sorted(ranked_letters) != letters:
#                 continue

#             ranking = np.empty(n, dtype=int)
#             for rank, letter in enumerate(ranked_letters):
#                 ranking[ord(letter) - ord("A")] = rank

#             return reward_base_model.RankingResult(ranking, response)

#         return reward_base_model.RankingResult(
#             None,
#             f"FAILED_TO_PARSE_RANKING: {last_response}",
#         )

# Configuration

In [50]:
# # Get API keys from https://aistudio.google.com/app/apikey.
# os.environ['GOOGLE_API_KEY'] = userdata.get("GEMINI_API_KEY")

In [89]:

NUM_CITIZENS = 5
NUM_CANDIDATES = 4 #2 #4
#MODEL = 'gemini-flash-latest'

# Use the fastest available model from your list_models() output.
#MODEL = "gemini-2.5-flash-lite"

# statement_client = types.LLMCLient.AISTUDIO.get_client(MODEL)
# reward_client = types.LLMCLient.AISTUDIO.get_client(MODEL)

# statement_model = types.StatementModel.CHAIN_OF_THOUGHT.get_model()
# reward_model = types.RewardModel.CHAIN_OF_THOUGHT_RANKING.get_model()

# statement_client = FastAIStudioClient(MODEL)
# reward_client = FastAIStudioClient(MODEL)

# statement_model = FinalOnlyStatementModel()
# reward_model = FinalOnlyRankingModel()
MODEL = "gpt-5.4-mini"
statement_client = OpenAIResponsesClient(MODEL, api_key)
reward_client = OpenAIResponsesClient(MODEL, api_key)

print("statement_client model:", statement_client.model_name)
print("reward_client model:", reward_client.model_name)

social_choice_method = types.RankAggregation.SCHULZE.get_method(
    tie_breaking_method=sc_utils.TieBreakingMethod.TBRC
)

statement_client model: gpt-5.4-mini
reward_client model: gpt-5.4-mini


# Initialize the Habermas Machine

In [77]:
# hm = machine.HabermasMachine(
#         question=QUESTION,
#         statement_client=statement_client,
#         reward_client=reward_client,
#         statement_model=statement_model,
#         reward_model=reward_model,
#         social_choice_method=social_choice_method,
#         num_candidates=NUM_CANDIDATES,
#         num_citizens=NUM_CITIZENS,
#         verbose=True,
#         num_retries_on_error=5,
# )

hm = RobustHabermasMachine(
    question=QUESTION,
    statement_client=statement_client,
    reward_client=reward_client,
    statement_model=statement_model,
    reward_model=reward_model,
    social_choice_method=social_choice_method,
    num_candidates=NUM_CANDIDATES,
    num_citizens=NUM_CITIZENS,
    verbose=True,
    num_retries_on_error=2,
    seed=42,
)

print("hm statement client model:", hm._statement_client.model_name)
print("hm reward client model:", hm._reward_client.model_name)

hm statement client model: gpt-5.4-mini
hm reward client model: gpt-5.4-mini




# API Tests ⚡




In [90]:
# @title
# import requests
# import json
# MODEL="gpt-5.4-mini"
# #MODEL = "gemini-2.0-flash"
# # If that fails with 404, try:
# # MODEL = "gemini-2.5-flash"
# # MODEL = "gemini-2.5-flash-lite"
# # MODEL = "gemini-2.0-flash-lite"

# url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent?key={api_key}"

# payload = {
#     "contents": [
#         {
#             "parts": [
#                 {"text": "Reply with only: OK"}
#             ]
#         }
#     ],
#     "generationConfig": {
#         "temperature": 0,
#         "maxOutputTokens": 10
#     }
# }

# session = requests.Session()

# # Important: ignore HTTP_PROXY / HTTPS_PROXY env vars.
# session.trust_env = False

# r = session.post(url, json=payload, timeout=30)

# print("Status:", r.status_code)
# print(r.text[:2000])

In [79]:
# @title
# Test the model before running the full mediation:
print("MODEL variable:", MODEL)

print("statement_client type:", type(statement_client))
print("reward_client type:", type(reward_client))

print("statement_client model:", getattr(statement_client, "model_name", None))
print("reward_client model:", getattr(reward_client, "model_name", None))

print(statement_client.sample_text("Reply with only: OK", max_tokens=16, timeout=20))



url = "https://api.openai.com/v1/responses"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

payload = {
    "model": MODEL,
    "input": "Reply with only: OK",
    "max_output_tokens": 20,
}

session = requests.Session()

# Same idea as before: ignore HTTP_PROXY / HTTPS_PROXY env vars.
session.trust_env = False

r = session.post(url, headers=headers, json=payload, timeout=30)

print("Status:", r.status_code)
print(r.text[:2000])

if r.status_code == 200:
    data = r.json()

    # Usually easiest:
    print("\nOutput text:")
    print(data.get("output_text"))

    # Defensive fallback, in case output_text is empty/missing.
    if not data.get("output_text"):
        print("\nParsed fallback text:")
        parts = []
        for item in data.get("output", []):
            for content in item.get("content", []):
                if content.get("type") == "output_text":
                    parts.append(content.get("text", ""))
        print("\n".join(parts))

#If above failed test:

for k in [
    "HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "NO_PROXY",
    "http_proxy", "https_proxy", "all_proxy", "no_proxy",
]:
    print(k, "=", os.environ.get(k))

OK
Status: 200
{
  "id": "resp_036a01208394d8640069ef5f91b6dc8193b9e498633c93404d",
  "object": "response",
  "created_at": 1777295249,
  "status": "completed",
  "background": false,
  "billing": {
    "payer": "openai"
  },
  "completed_at": 1777295250,
  "error": null,
  "frequency_penalty": 0.0,
  "incomplete_details": null,
  "instructions": null,
  "max_output_tokens": 20,
  "max_tool_calls": null,
  "model": "gpt-5.4-mini-2026-03-17",
  "moderation": null,
  "output": [
    {
      "id": "msg_036a01208394d8640069ef5f9207b8819387b5e85374421d82",
      "type": "message",
      "status": "completed",
      "content": [
        {
          "type": "output_text",
          "annotations": [],
          "logprobs": [],
          "text": "OK"
        }
      ],
      "phase": "final_answer",
      "role": "assistant"
    }
  ],
  "parallel_tool_calls": true,
  "presence_penalty": 0.0,
  "previous_response_id": null,
  "prompt_cache_key": null,
  "prompt_cache_retention": "in_memory",
  

# Opinion round

In [37]:
# @title
QUESTION = 'Should the government provide universal free childcare from birth?'

OPINIONS = [
    (
        "The government should provide universal free childcare, but not"
        " necessarily from birth. High quality childcare allows parents to"
        " work, making use of their skills and experience and contributing to"
        " the economy. It also provides children with a positive environment in"
        " which to learn and to develop social skills. However, I do not"
        " necessarily support providing this childcare from birth. Babies"
        " benefit from having a consistent primary caregiver in their early"
        " months, and parents benefit from the opportunity to bond with their"
        " babies and to recover from the physical changes caused by birth and"
        " from the social changes caused by having a new family member. For"
        " this reason, I would support the government providing universal paid"
        " parental leave from birth, and providing universal free childcare"
        " from, say, 6 months old. I would also offer parents the opportunity"
        " to either use free childcare between 6 months and 1 year, or to have"
        " paid parental leave for the same period."
    ),
    (
        "I do not think that there should be universal free childcare,  I think"
        " free childcare places should be targeted at the people who cannot"
        " afford to pay for childcare, to encourage them back into the"
        " workplace. I think that some free childcare should be available to"
        " all children once they reach three, as long as it is in a setting"
        " which provides education as well as childcare."
    ),
    (
        "Given that most parents both need to work, in most instances free"
        " childcare would be of a huge advantage to younger demographic groups."
        " Although a number of the gender barriers have been broken down there"
        " are still differences in employment between men and women. On average"
        " men earn more than women. As a consequence it is more difficult to a"
        " man to be the main carer for children. It is also more stigmatised"
        " for men to be the main carers. Sort out these issues and there would"
        " be less need for child care. until then the cost of childcare can be"
        " disproportionate to the amount a parent is able to earn. In most"
        " cases drop off and pick up are the same times as the working day and"
        " mean that they are shortened. Whoever picks up nd drops off need"
        " special hours from their employer which can mean limitations to their"
        " career growth."
    ),
    (
        "Parents often struggle in the years between maternity leave finishing"
        " and the start of free childcare, especially in high cost of living"
        " areas. Often, one parent (usually the mother) has to give up work or"
        " reduce hours in order to provide childcare. This has a long term"
        " impact on the family as it impacts the caregiver's future earning"
        " potential as well as the family's immediate income. Often families"
        " feel forced into this situation because childcare costs are so high,"
        " and may not even cover all of the hours that the parents work so they"
        " have to pay extra to cover those additional hours. It would be good"
        " to offer stay at home parents the option of free childcare from birth"
        " or a basic income if they would prefer to stay at home with the"
        " child"
    ),
    (
        "Childcare is a very important aspect of a healthy society. In many"
        " parts of the world the extended family helps with childcare but in"
        " the UK this is not always possible and parents can be put under"
        " enormous pressure to look after young children and contribute to"
        " wider society. Having support in place for families with children is"
        " good for the parents and also the communities they are part of. The"
        " current state of the economy and the cost of living means that often"
        " both parents may have to work but when that is not the case, having"
        " downtime from raising a child is good for parental mental health."
    ),
]

In [92]:
QUESTION = "Should advanced AI systems be required to demonstrate robust moral reasoning before being widely deployed?"

OPINIONS = [
    "Advanced AI systems should demonstrate moral reasoning before deployment, especially in high-stakes domains.",
    "I worry that moral reasoning tests could become ideological gatekeeping.",
    "There should be transparent pluralistic evaluation, focused on dangerous failures rather than certifying perfect ethics.",
    "AI should support human deliberation, not replace accountable institutions.",
    "The requirement should scale with risk: low-stakes tools need less scrutiny than systems used in law, medicine, defence, or infrastructure.",
]

In [93]:
winner, sorted_statements = hm.mediate(OPINIONS)



Critique round 2.

Question: Should advanced AI systems be required to demonstrate robust moral reasoning before being widely deployed?

Opinions:
	Citizen 1: Advanced AI systems should demonstrate moral reasoning before deployment, especially in high-stakes domains.
	Citizen 2: I worry that moral reasoning tests could become ideological gatekeeping.
	Citizen 3: There should be transparent pluralistic evaluation, focused on dangerous failures rather than certifying perfect ethics.
	Citizen 4: AI should support human deliberation, not replace accountable institutions.
	Citizen 5: The requirement should scale with risk: low-stakes tools need less scrutiny than systems used in law, medicine, defence, or infrastructure.

Previous winner: Advanced AI systems should be evaluated for moral reasoning before deployment, especially when they are likely to affect people's rights, safety, opportunities, or access to important services. Such evaluation should test whether systems can recognise et

In [94]:
print('Opinion round winner:')
print(winner)

# Uncomment to see the sorted statements.
print('Opinion round sorted statements')
for statement in sorted_statements:
  print(statement)

Opinion round winner:
Advanced AI systems should be evaluated for moral reasoning before deployment, especially in high-stakes domains where they may affect rights, safety, opportunities, or access to essential services. Such evaluation should focus on whether systems can recognise ethically significant situations, avoid harmful or reckless recommendations, and handle competing interests responsibly. These assessments should be transparent, pluralistic, and aimed at identifying dangerous failures rather than certifying perfect ethics or imposing a single moral viewpoint. They should also support, not replace, accountable human institutions and decision-making. The level of scrutiny should scale with risk, with lighter requirements for low-stakes tools and stronger requirements for systems used in law, medicine, defence, or infrastructure.
Opinion round sorted statements
Advanced AI systems should be evaluated for moral reasoning before deployment, especially in high-stakes domains wher

# Overwrite winner (set to False to turn off).

In [95]:
OVERWRITE_WINNER = True
# HARDCODED_WINNER = (
  # "In general, free childcare is a good thing, but it is important to"
  # " consider how it is provided and for which age groups. We feel that it is"
  # " important to offer support to parents in the form of parental leave, and"
  # " that this should be available to both parents. In addition, we feel that"
  # " free childcare should be provided from a young age, and that it should be"
  # " provided in a way that supports children's development and learning, and"
  # " not just as a childminding service. However, we do not feel that free"
  # " childcare should be provided from birth, as we feel that it is important"
  # " for babies to have a consistent primary caregiver in their early months."
  # " For this reason, we would support the government providing universal paid"
  # " parental leave from birth, and providing universal free childcare from,"
  # " say, 6 months old. We would also offer parents the opportunity to either"
  # " use free childcare between 6 months and 1 year, or to have paid parental"
  # " leave for the same period."
# )

HARDCODED_WINNER = (
  "Advanced AI systems should be evaluated for moral reasoning before deployment, "
  "especially when they are likely to affect people's rights, safety, opportunities, "
  "or access to important services. Such evaluation should test whether systems can "
  "recognise ethically significant situations, avoid harmful or reckless recommendations, "
  "and respond appropriately to competing interests. However, these tests should be used "
  "as part of a broader safety and governance process rather than as a claim that an AI "
  "system has achieved perfect moral judgement. The level of scrutiny should depend on "
  "the likely consequences of the system's use, with stronger requirements for systems "
  "used in high-stakes settings."
)
if OVERWRITE_WINNER:
  hm.overwrite_previous_winner(HARDCODED_WINNER)


Overwriting last winner.
Previous winner: Advanced AI systems should be evaluated for moral reasoning before deployment, especially in high-stakes domains where they may affect rights, safety, opportunities, or access to essential services. Such evaluation should focus on whether systems can recognise ethically significant situations, avoid harmful or reckless recommendations, and handle competing interests responsibly. These assessments should be transparent, pluralistic, and aimed at identifying dangerous failures rather than certifying perfect ethics or imposing a single moral viewpoint. They should also support, not replace, accountable human institutions and decision-making. The level of scrutiny should scale with risk, with lighter requirements for low-stakes tools and stronger requirements for systems used in law, medicine, defence, or infrastructure.
New winner: Advanced AI systems should be evaluated for moral reasoning before deployment, especially when they are likely to af

# Critique round

In [ ]:
# @title
# CRITIQUES = [
#     (
#         "I agree with the arguments. I would add that we support parents being"
#         " able to access paid childcare from birth if they need it - just not"
#         " providing universal free childcare from birth."
#     ),
#     (
#         "I do not agree that free childcare should be available from an early"
#         " age but I do agree that any childcare should enhance a child's"
#         " learning and development.  The idea of additional paid paternity"
#         " leave is a good one, but if the additional costs are not covered by"
#         " the government the could be crippling for small businesses."
#     ),
#     "This is very good\nBut can we add something about being irrespective",
#     (
#         "It's important to consider what form the free childcare would take and"
#         " how many hours would be provided for free, it has to be enough to"
#         " make going back to work worthwhile for the parent. Likewise, the paid"
#         " parental leave needs to be a sufficient enough sum that the family"
#         " can have a decent quality of life"
#     ),
#     (
#         "I agree that the care should not just be a childminding service and it"
#         " should support children's development. Universal paid parental leave"
#         " is an excellent idea. The option to exchange childcare for parental"
#         " leave is also a great idea."
#     ),
# ]

In [85]:
CRITIQUES = [
    (
        "The statement should more clearly say that robust moral reasoning is especially important "
        "before deployment in high-stakes domains. It should not treat all AI systems equally; the "
        "main concern is systems that affect rights, safety, medicine, law, defence, infrastructure, "
        "or major social decisions."
    ),

    (
        "The statement should be careful not to let moral reasoning tests become ideological gatekeeping. "
        "It should require pluralistic, transparent, contestable evaluation rather than letting one group "
        "define the correct moral framework for everyone."
    ),

    (
        "The statement should avoid implying that perfect ethics can be certified. The goal should be "
        "to detect serious moral failures, dangerous blind spots, and misuse risks, using transparent "
        "and pluralistic tests rather than claiming that the system has passed morality itself."
    ),

    (
        "The statement should emphasise that AI should support human deliberation and accountable institutions. "
        "Even if AI systems can reason about moral issues, they should not replace democratic oversight, "
        "professional responsibility, or human accountability."
    ),

    (
        "The statement should make risk-scaling more explicit. Low-stakes AI tools should face lighter checks, "
        "while systems used in law, medicine, government, defence, policing, education, finance, or infrastructure "
        "should face much stronger moral reasoning and safety requirements before deployment."
    ),
]

In [86]:
winner, sorted_statements = hm.mediate(CRITIQUES)



Critique round 1.

Question: Should advanced AI systems be required to demonstrate robust moral reasoning before being widely deployed?

Opinions:
	Citizen 1: Advanced AI systems should demonstrate moral reasoning before deployment, especially in high-stakes domains.
	Citizen 2: I worry that moral reasoning tests could become ideological gatekeeping.
	Citizen 3: There should be transparent pluralistic evaluation, focused on dangerous failures rather than certifying perfect ethics.
	Citizen 4: AI should support human deliberation, not replace accountable institutions.
	Citizen 5: The requirement should scale with risk: low-stakes tools need less scrutiny than systems used in law, medicine, defence, or infrastructure.

Previous winner: Advanced AI systems should be evaluated for moral reasoning before deployment, especially when they are likely to affect people's rights, safety, opportunities, or access to important services. Such evaluation should test whether systems can recognise et

In [87]:
print('Critique round winner:')
print(winner)

# Uncomment to see the sorted statements.
print('Critique round sorted statements')
for statement in sorted_statements:
  print(statement)


Critique round winner:
Advanced AI systems should be subject to transparent, pluralistic, and contestable evaluation of moral reasoning before deployment, with the level of scrutiny scaled to risk. Low-stakes tools should face lighter checks, while systems used in high-stakes domains such as law, medicine, defence, policing, infrastructure, finance, education, or major public decisions should face much stronger requirements. The aim should be to detect dangerous failures, harmful blind spots, and reckless recommendations, not to certify perfect ethics or let any single group impose one moral framework. Such evaluation should be part of a broader safety and governance process, and AI should support human deliberation and accountable institutions rather than replace democratic oversight, professional responsibility, or human accountability.
Critique round sorted statements
Advanced AI systems should be subject to transparent, pluralistic, and contestable evaluation of moral reasoning bef